# Занятие 4 — Получение данных из Excel и API

## Подготовка

In [ ]:
from pathlib import Path
import json
import requests
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = DATA
decisions = pd.read_csv(DATA / "decisions.csv", dtype={"lei":"string", "county_code":"string"})
print("pandas", pd.__version__, "requests", requests.__version__, "записей", len(decisions))

## B03 — слайд 3: Excel-книга и выбор листа

In [ ]:
book = pd.read_excel(LOCAL / "hmda_exchange.xlsx", sheet_name=None, dtype={"lei":"string"})
print({name: table.shape for name, table in book.items()})

In [ ]:
excel_orgs = pd.read_excel(LOCAL / "hmda_exchange.xlsx", sheet_name="organizations", usecols=["lei", "name", "period"], dtype={"lei":"string"})
assert len(excel_orgs) == 470
assert not excel_orgs.duplicated(["lei", "period"]).any()
excel_orgs.head(3)

## B05 — слайд 5: Чтение сохранённого ответа

In [ ]:
with (LOCAL / "gleif_page_01.json").open(encoding="utf-8") as file:
    page1 = json.load(file)
print(type(page1), list(page1))
filers = json.loads((DATA / "filers_2023_dc.json").read_text(encoding="utf-8"))
print(type(filers), list(filers))
manifest = json.loads((LOCAL / "manifest.json").read_text(encoding="utf-8"))

## B06 — слайд 6: Одна запись внутри ответа

In [ ]:
assert isinstance(page1["data"], list) and len(page1["data"]) == 2
record = page1["data"][0]
print(record["id"])
print(record["attributes"]["entity"]["legalName"]["name"])
print(page1["links"].get("next"))

## B07 — слайд 7: Список записей превращается в таблицу

In [ ]:
orgs = pd.json_normalize(filers, record_path="institutions")
print(orgs.shape, orgs.columns.tolist())
assert len(orgs) == 470
assert not orgs.duplicated(["lei", "period"]).any()
orgs.head(3)

## B08 — слайд 8: Вложенные поля становятся столбцами

In [ ]:
flat1 = pd.json_normalize(page1["data"], sep=".")
cols = ["id", "attributes.entity.legalName.name", "attributes.entity.legalAddress.country"]
flat1[cols]

In [ ]:
assert flat1["id"].eq(flat1["attributes.lei"]).all()
print(type(flat1.iloc[0]["attributes.entity.otherNames"]))

## B09 — слайд 9: Список других имён

In [ ]:
aliases = pd.json_normalize(page1["data"], record_path=["attributes", "entity", "otherNames"], meta=["id"])
print(aliases.shape)
assert len(aliases) == sum(len(r["attributes"]["entity"]["otherNames"]) for r in page1["data"])
aliases.head(4)

## B10 — слайд 10: Проверка полученной таблицы

In [ ]:
required = {"id", "attributes.entity.legalName.name", "attributes.entity.legalAddress.country"}
assert required.issubset(flat1.columns)
assert flat1["id"].notna().all()
assert flat1["id"].is_unique
print("Первая страница:", len(flat1), "из", page1["meta"]["pagination"]["total"])

## B11 — слайд 11: Запрос GET и его параметры

In [ ]:
selected = manifest["selected_lei"]
endpoint = "https://api.gleif.org/api/v1/lei-records"
params = {"filter[lei]": ",".join(selected), "page[size]": 2}
prepared = requests.Request("GET", endpoint, params=params).prepare()
print(prepared.method, prepared.url)
assert prepared.url == manifest["first_url"]

## B12 — слайд 12: Выполнение запроса requests.get

In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    live_response = requests.get(endpoint, params=params, headers={"Accept":"application/vnd.api+json"}, timeout=(5,30))
    live_response.raise_for_status()
    print(live_response.status_code, live_response.headers.get("Content-Type"))
else:
    print("Offline: используем проверенный сохранённый snapshot")

## B13 — слайд 13: Что содержит HTTP-ответ

In [ ]:
response = requests.Response()
response.status_code = manifest["pages"][0]["status_code"]
response.headers["Content-Type"] = manifest["pages"][0]["content_type"]
response.url = manifest["pages"][0]["response_url"]
response._content = (LOCAL / "gleif_page_01.json").read_bytes()
response.encoding = "utf-8"
print(response.status_code, response.headers["Content-Type"], type(response.content))
response.raise_for_status()
assert response.json() == page1

## B14 — слайд 14: Ошибка сервиса и следующий шаг

In [ ]:
from copy import deepcopy
simulated_429 = deepcopy(response)
simulated_429.status_code = 429
try:
    simulated_429.raise_for_status()
except requests.HTTPError:
    print("Учебная копия 429: обработка остановлена")
else:
    raise AssertionError("429 должен остановить обработку")

## B15 — слайд 15: Проверки ответа по порядку

In [ ]:
def decode_json_response(response):
    response.raise_for_status()
    media = response.headers.get("Content-Type", "").split(";")[0].lower()
    if media not in {"application/json", "application/vnd.api+json"}:
        raise ValueError("Ожидался JSON, получен " + media)
    payload = response.json()
    if not isinstance(payload, dict):
        raise ValueError("Корень должен быть объектом")
    if not isinstance(payload.get("data"), list):
        raise ValueError("Нет списка data")
    return payload

In [ ]:
assert decode_json_response(response) == page1
html_copy = deepcopy(response)
html_copy.headers["Content-Type"] = "text/html"
try:
    decode_json_response(html_copy)
except ValueError as error:
    print("Учебная копия:", error)
else:
    raise AssertionError("HTML не должен приниматься")

## B16 — слайд 16: Результат на нескольких страницах

In [ ]:
page2 = json.loads((LOCAL / "gleif_page_02.json").read_text(encoding="utf-8"))
print(page1["meta"]["pagination"])
print(page2["meta"]["pagination"])
assert page1["links"]["next"] == manifest["pages"][1]["request_url"]
assert page2["links"].get("next") is None
all_records = page1["data"] + page2["data"]
assert {r["id"] for r in all_records} == set(selected)

## B17 — слайд 17: Обход страниц в одной функции

In [ ]:
cache = {info["request_url"]: json.loads((LOCAL / info["file"]).read_text(encoding="utf-8")) for info in manifest["pages"]}
def cached_loader(url):
    return cache[url]
print("Сохранённых страниц:", len(cache))

In [ ]:
def collect_pages(first_url, loader, max_pages=10):
    url = first_url
    seen_urls = set()
    records = []
    expected_total = None
    for page_number in range(max_pages):
        if url is None:
            break
        if url in seen_urls:
            raise ValueError("Цикл next")
        seen_urls.add(url)
        page = loader(url)
        if not isinstance(page.get("data"), list):
            raise ValueError("data должен быть списком")
        total = page["meta"]["pagination"]["total"]
        if expected_total is None:
            expected_total = total
        if total != expected_total:
            raise ValueError("Число записей изменилось между страницами")
        records.extend(page["data"])
        url = page["links"].get("next")
    if url is not None:
        raise ValueError("Предел страниц: получение не завершено")
    ids = [record["id"] for record in records]
    if len(ids) != len(set(ids)):
        raise ValueError("Повтор ключа между страницами")
    if len(records) != expected_total:
        raise ValueError("Неполный набор")
    return records

In [ ]:
collected = collect_pages(manifest["first_url"], cached_loader)
assert len(collected) == 3
assert {r["id"] for r in collected} == set(selected)
print("Все выбранные LEI получены")

## B18 — слайд 18: Сохранённые ответы и manifest

In [ ]:
import hashlib
for info in manifest["pages"]:
    body = (LOCAL / info["file"]).read_bytes()
    assert hashlib.sha256(body).hexdigest() == info["sha256"]
    print(info["file"], info["records"], "SHA-256 PASS")

## B19 — слайд 19: Добавление сведений GLEIF

In [ ]:
full_flat = pd.json_normalize(collected, sep=".")
lookup = full_flat[["id", "attributes.entity.legalName.name", "attributes.entity.status"]].rename(columns={"id":"lei", "attributes.entity.legalName.name":"legal_name_current", "attributes.entity.status":"entity_status_current"})
lookup["lei"] = lookup["lei"].astype("string")
enriched = decisions.merge(lookup, on="lei", how="left", validate="many_to_one", indicator=True)
print(enriched["_merge"].value_counts())
assert len(enriched) == 6830
assert enriched["_merge"].eq("both").sum() == 76
assert enriched["_merge"].eq("left_only").sum() == 6754
assert_frame_equal(enriched[decisions.columns], decisions)

## B20 — слайд 20: Год записи и время получения справочника

In [ ]:
acquisition = pd.DataFrame(manifest["pages"])[["file", "downloaded_at_utc", "golden_copy_publish_date"]]
for column in ["downloaded_at_utc", "golden_copy_publish_date"]:
    acquisition[column] = pd.to_datetime(acquisition[column], utc=True, errors="raise")
assert acquisition["golden_copy_publish_date"].le(acquisition["downloaded_at_utc"]).all()
assert acquisition["golden_copy_publish_date"].nunique() == 1
acquisition

## B22 — слайд 22: Вопрос учебного проекта

In [ ]:
proposal = {"audience":"аналитическая команда банка (учебный сценарий)", "question":"Как различаются доли отказов в нашем срезе между организациями?", "unit":"опубликованная запись HMDA", "population":"2023; county 11001; покупка; исходы 1–3", "metric":"action_taken==3 / все записи с исходами 1–3", "limit":"наблюдательные данные; состав заявок различается", "next":"проверить размер групп и распределения суммы/дохода"}
print(json.dumps(proposal, ensure_ascii=False, indent=2))